# 中文情緒分類：資料處理與 BERT 訓練

使用中文句子及其情緒標籤，微調 `google-bert/bert-base-chinese`。
本 notebook 負責讀取 CSV、清理資料、切分訓練／驗證集、訓練模型與匯出成果。
完成後，開啟 `predict.ipynb` 預測句子並繪製訓練圖表。

## 快速開始

1. 將此 notebook 上傳至 Google Colab。
2. 在「執行階段 → 變更執行階段類型」選擇 GPU。
3. 從上往下執行，並在「設定專案位置」選擇暫存空間或 Google Drive。
4. 準備 `data.csv`；若專案目錄內沒有此檔，程式會顯示上傳按鈕。
5. 訓練完成後，暫存空間模式會下載 `bert_artifacts.zip`；Drive 模式會將成果留在專案資料夾。

## 資料格式

CSV 必須依序包含 `text`、`emotion` 兩欄，文字使用 UTF-8 編碼：

```csv
text,emotion
你要不要去吃午餐？,平淡語氣
誒誒誒！我甄選上了！,開心語調
```

資料來源：[Chinese Multi-Emotion Dialogue Dataset](https://huggingface.co/datasets/Johnson8187/Chinese_Multi-Emotion_Dialogue_Dataset)。
類別數與情緒名稱會由資料自動建立，不需手動填寫。


## 1. 安裝與匯入套件

執行安裝儲存格後再匯入套件。PyTorch 使用 Colab 提供的環境。
使用 Transformers 5.20.0，並將 huggingface-hub 限制為 `>=1.31,<2.0`，相容於環境中的 Diffusers 0.40.0 與 Gradio 6.29.0。
首次安裝或變更套件版本後，請選擇「執行階段 → 重新啟動工作階段」，再從下方匯入套件儲存格繼續。


In [ ]:
%pip install -q --upgrade "transformers==5.20.0" "huggingface-hub>=1.31,<2.0" "datasets>=3,<5" "accelerate>=1.2,<2" scikit-learn


In [ ]:
import csv
import json
from collections import Counter
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)


## 2. 設定專案位置

選擇一種儲存方式，兩份 notebook 必須使用相同設定：

| 設定 | 專案位置 | 使用方式 |
| --- | --- | --- |
| `USE_GOOGLE_DRIVE = False` | `/content/HW_bert_finetune` | 使用 Colab 暫存空間，以壓縮檔帶走模型與紀錄 |
| `USE_GOOGLE_DRIVE = True` | `/content/drive/MyDrive/HW_bert_finetune` | 掛載 Google Drive，將資料與輸出直接存入雲端硬碟 |

使用 Drive 時，執行下方儲存格並完成授權。若要更換資料夾，可修改 `PROJECT_DIR`。
Colab 暫存空間會隨執行階段刪除，請在結束前下載產出，或使用 Drive 儲存。
詳見 [Colab 說明](https://research.google.com/colaboratory/faq.html)。


In [ ]:
USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/HW_bert_finetune")
else:
    PROJECT_DIR = Path("/content/HW_bert_finetune") if Path("/content").is_dir() else Path.cwd()
    if (PROJECT_DIR / "HW_bert_finetune").is_dir():
        PROJECT_DIR /= "HW_bert_finetune"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print(f"專案位置：{PROJECT_DIR}")


## 3. 資料與訓練參數

以固定隨機種子將資料分為 80% 訓練集與 20% 驗證集，各情緒在兩組中的比例會盡量一致。
模型每 30 個 optimizer steps 驗證並儲存；以驗證 loss 最低的 checkpoint 作為最佳模型。

| 參數 | 預設值 | 用途 |
| --- | --- | --- |
| `NUM_TRAIN_EPOCHS` | 3 | 訓練資料的遍歷次數 |
| `TRAIN_BATCH_SIZE` / `EVAL_BATCH_SIZE` | 32 / 32 | 每批訓練／驗證筆數 |
| `GRADIENT_ACCUMULATION_STEPS` | 2 | 累積兩批梯度後更新參數 |
| `LEARNING_RATE` | `1e-4` | 學習率 |
| `MAX_SEQ_LENGTH` | 512 | 每句最多使用的 token 數 |
| `EARLY_STOPPING_PATIENCE` | 3 | 連續三次驗證 loss 未改善時停止 |

CUDA GPU 會啟用 FP16；沒有 GPU 時可使用 CPU，但訓練較慢。
若出現 GPU 記憶體不足，可將 batch size 降為 16 或 8。
要保留不同實驗的成果，請在訓練前修改 `MODEL_DIR`。


In [ ]:
INPUT_FILE = PROJECT_DIR / "data.csv"
DATA_DIR = PROJECT_DIR / "data/bert"
MODEL_DIR = PROJECT_DIR / "output"

MODEL_NAME = "google-bert/bert-base-chinese"
SEED = 42
VALID_RATIO = 0.2
MAX_SEQ_LENGTH = 512
NUM_TRAIN_EPOCHS = 3
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 1e-4
WARMUP_STEPS = 50
WEIGHT_DECAY = 0.01
EVAL_STEPS = 30
SAVE_TOTAL_LIMIT = 2
EARLY_STOPPING_PATIENCE = 3
LOGGING_STEPS = 10

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"訓練裝置：{device_name}")
print(f"模型輸出：{MODEL_DIR}")


## 4. 上傳與清理資料

已放入專案資料夾的 `data.csv` 會直接使用；缺少檔案時，請在上傳視窗選擇 `data.csv`。
程式會移除文字與標籤兩端的空白，略過空白文字，並檢查欄位數及空白標籤。
情緒名稱依排序轉為整數 ID，這份對照會同時存入資料檔與模型設定。


In [ ]:
if not INPUT_FILE.is_file():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"請將 data.csv 放在 {PROJECT_DIR}") from None
    print("請上傳 data.csv")
    uploaded = files.upload()
    if "data.csv" not in uploaded:
        raise FileNotFoundError("未收到 data.csv，請重新執行此儲存格並選擇該檔案。")
    INPUT_FILE.write_bytes(uploaded["data.csv"])
    del uploaded


In [ ]:
def load_data(path):
    """讀取 CSV，檢查格式並略過沒有文字的記錄。"""
    texts, emotions = [], []
    skipped = 0
    with path.open(encoding="utf-8-sig", newline="") as file:
        reader = csv.reader(file)
        if next(reader, None) != ["text", "emotion"]:
            raise ValueError("CSV 欄位必須依序為 text、emotion。")
        for row_number, row in enumerate(reader, start=2):
            if not row:
                continue
            if len(row) != 2:
                raise ValueError(f"第 {row_number} 筆 CSV 記錄的欄位數不是 2：{row}")
            text, emotion = (value.strip() for value in row)
            if not text:
                skipped += 1
                continue
            if not emotion:
                raise ValueError(f"第 {row_number} 筆 CSV 記錄的 emotion 不可為空。")
            texts.append(text)
            emotions.append(emotion)
    if not texts:
        raise ValueError(f"{path} 沒有有效資料。")
    if skipped:
        print(f"已略過 {skipped} 筆空白文字。")
    return texts, emotions


texts, emotions = load_data(INPUT_FILE)
LABEL2ID = {name: index for index, name in enumerate(sorted(set(emotions)))}
ID2LABEL = {index: name for name, index in LABEL2ID.items()}
labels = [LABEL2ID[name] for name in emotions]
if len(LABEL2ID) < 2 or min(Counter(labels).values()) < 2:
    raise ValueError("分層切分至少需要兩種情緒，且每種情緒至少有兩筆有效資料。")


## 5. 切分與匯出資料

產生 `data/bert/train.jsonl`、`valid.jsonl` 與 `label_mapping.json`。
每筆 JSONL 包含 `sentences`（文字）與 `labels`（整數 ID）。
固定 seed 可重現相同的切分結果，方便比較不同訓練參數。


In [ ]:
train_texts, valid_texts, train_labels, valid_labels = train_test_split(
    texts, labels, test_size=VALID_RATIO, random_state=SEED, stratify=labels
)
splits = {
    "train": {"sentences": train_texts, "labels": train_labels},
    "validation": {"sentences": valid_texts, "labels": valid_labels},
}

DATA_DIR.mkdir(parents=True, exist_ok=True)
for split_name, filename in (("train", "train.jsonl"), ("validation", "valid.jsonl")):
    split = splits[split_name]
    with (DATA_DIR / filename).open("w", encoding="utf-8") as file:
        for text, label in zip(split["sentences"], split["labels"]):
            file.write(json.dumps({"sentences": text, "labels": label}, ensure_ascii=False) + "\n")
with (DATA_DIR / "label_mapping.json").open("w", encoding="utf-8") as file:
    json.dump({"label2id": LABEL2ID, "id2label": ID2LABEL}, file, ensure_ascii=False, indent=2)

train_counts, valid_counts = Counter(train_labels), Counter(valid_labels)
print(f"有效資料：{len(texts)}；訓練：{len(train_labels)}；驗證：{len(valid_labels)}")
for label, name in ID2LABEL.items():
    print(f"  {label} {name}：train={train_counts[label]}，valid={valid_counts[label]}")
print(f"資料輸出：{DATA_DIR}")


## 6. Tokenization 與模型初始化

第一次執行會下載 tokenizer 與基礎模型。
每句文字最多保留 `MAX_SEQ_LENGTH` 個 tokens，再於組成 batch 時動態補齊長度，減少 padding 計算。
分類層的類別數與情緒名稱由前面建立的標籤對照決定。


In [ ]:
set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
dataset = DatasetDict({name: Dataset.from_dict(split) for name, split in splits.items()})


def tokenize_batch(batch):
    return tokenizer(batch["sentences"], truncation=True, max_length=MAX_SEQ_LENGTH)


tokenized_dataset = dataset.map(tokenize_batch, batched=True, remove_columns=["sentences"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABEL2ID), label2id=LABEL2ID, id2label=ID2LABEL
)


## 7. 設定評估指標與訓練流程

Macro F1 先計算每種情緒的 F1，再取平均，讓各類別具有相同權重。
訓練紀錄中的 `eval_f1` 會用於 `predict.ipynb` 的 F1 曲線。
最佳模型與提前停止皆依 `eval_loss` 判斷；loss 越低越好。

詳見 [Hugging Face Trainer 文件](https://huggingface.co/docs/transformers/en/main_classes/trainer)。


In [ ]:
def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = logits.argmax(axis=-1)
    return {"f1": f1_score(eval_pred.label_ids, predictions, average="macro", zero_division=0)}


training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="linear",
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_steps=EVAL_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    logging_steps=LOGGING_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    bf16=False,
    tf32=False,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)


## 8. 訓練與儲存最佳模型

此步驟開始微調模型，完成時間取決於資料量及 GPU。
輸出至 `MODEL_DIR` 的檔案包括模型權重、`config.json`、tokenizer 與 `trainer_state.json`。
根目錄中的模型是依驗證 loss 選出的最佳模型；`checkpoint-*` 是訓練期間的檢查點。


In [ ]:
trainer.train()

# 訓練結束時已載回最佳 checkpoint，匯出模型供獨立的預測階段使用。
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
trainer.save_state()
(MODEL_DIR / "label_mapping.json").write_text((DATA_DIR / "label_mapping.json").read_text(encoding="utf-8"), encoding="utf-8")

print(f"最佳 checkpoint：{trainer.state.best_model_checkpoint}")
print(f"最佳 eval_loss：{trainer.state.best_metric}")
print(f"模型與訓練紀錄：{MODEL_DIR}")


## 9. 保存訓練成果

暫存空間模式預設產生並下載 `bert_artifacts.zip`，包含最佳模型、tokenizer、訓練紀錄及切分後的資料；
中間 checkpoints 不會打包。壓縮模型可能需要一些時間。
在新的 Colab 執行階段開啟 `predict.ipynb` 時，上傳此壓縮檔即可還原。

Drive 模式不會自動下載，預測時使用相同的 Drive 專案位置即可。
若也需要本機備份，可將 `DOWNLOAD_ARTIFACTS` 改為 `True`。


In [ ]:
DOWNLOAD_ARTIFACTS = not USE_GOOGLE_DRIVE

if DOWNLOAD_ARTIFACTS:
    archive_path = PROJECT_DIR / "bert_artifacts.zip"
    artifact_files = [path for path in MODEL_DIR.iterdir() if path.is_file()]
    artifact_files += [DATA_DIR / name for name in ("train.jsonl", "valid.jsonl", "label_mapping.json")]
    with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
        for path in artifact_files:
            archive.write(path, arcname=path.relative_to(PROJECT_DIR).as_posix())
    print(f"訓練成果壓縮檔：{archive_path}")
    try:
        from google.colab import files
    except ImportError:
        pass
    else:
        files.download(str(archive_path))
else:
    print(f"訓練成果已儲存於：{MODEL_DIR}")
